# Thematic Analysis of Developer Feedback

This notebook processes and visualizes the thematic analysis of developer
feedback collected during the study.

The qualitative data were coded manually using **ATLAS.ti**. This notebook
does **not** perform the thematic coding itself. Instead, it processes the
ATLAS.ti export containing the coded quotes and summarizes the resulting
codes and themes.

The analysis performed here includes:

1. Loading and validating the ATLAS.ti export.
2. Summarizing the frequency of codes.
3. Summarizing the frequency of themes.
4. Inspecting the mapping between themes and codes.
5. Generating figures for reporting the thematic analysis.
6. Exporting aggregated results for reproducibility.


## 1. Setup


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ---------------------------------------------------------------------
# Display configuration
# ---------------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Change INPUT_FILE if the CSV is stored somewhere else.
#
# The output directory is intentionally separate from the input data so
# that generated tables and figures do not overwrite the source data.

INPUT_FILE = Path(
    "/content/drive/MyDrive/Documents/unb/thematic_analysis.csv"
)

OUTPUT_DIR = Path("/content/thematic_analysis_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Analysis configuration
# ---------------------------------------------------------------------

TOP_N_CODES = 10
TOP_N_CODES_PER_GROUP = 5

GROUP_LABELS = {
    "motivation": "Motivation",
    "tools": "Tools and Automation",
    "llm": "LLM Assistance",
}

# Keep the same colors across all figures.
COLORS = {
    "motivation": "#80B1D3",
    "tools": "#FDB462",
    "llm": "#8DD3C7",
}


# ---------------------------------------------------------------------
# Publication-oriented Matplotlib configuration
# ---------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")


In [ ]:
drive.mount("/content/drive")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file was not found: {INPUT_FILE}\n"
        "Update INPUT_FILE in the setup cell."
    )

df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df):,} rows.")
print(f"Columns: {list(df.columns)}")


## 2. Data Validation


In [ ]:
# Columns required by the subsequent analysis.
REQUIRED_COLUMNS = {"group", "code", "theme"}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise ValueError(
        "The ATLAS.ti export is missing required columns: "
        f"{sorted(missing_columns)}"
    )

# Basic dataset overview.
print("Dataset dimensions")
print(f"  Rows:    {len(df):,}")
print(f"  Columns: {len(df.columns):,}")

print("\nMissing values in analysis columns:")
display(
    df[list(REQUIRED_COLUMNS)]
    .isna()
    .sum()
    .rename("missing")
    .to_frame()
)

print("\nTheme groups:")
display(
    df["group"]
    .value_counts(dropna=False)
    .rename_axis("group")
    .to_frame("count")
)

unexpected_groups = (
    set(df["group"].dropna().unique()) - set(GROUP_LABELS)
)

if unexpected_groups:
    print(
        "\nWarning: unexpected groups found:",
        sorted(unexpected_groups),
    )
else:
    print("\nAll theme groups are recognized.")


## 3. Normalize Code Labels


The ATLAS.ti export contains one code with a long label that is normalized
for presentation. The normalization is performed **before** computing any
frequencies so that all subsequent analyses use the same labels.


In [ ]:
CODE_RENAMES = {
    "Preference for manual coding over LLM-assisted generation":
        "Preference for manual coding",
}

df_analysis = df.copy()

df_analysis["code"] = df_analysis["code"].replace(CODE_RENAMES)

print("Applied code-label normalization:")
for original, replacement in CODE_RENAMES.items():
    print(f"  {original!r} -> {replacement!r}")


## 4. Codes


In [ ]:
# Each row represents a coded excerpt. Counting rows therefore gives the
# frequency of each code in the ATLAS.ti export.

code_counts = (
    df_analysis
    .groupby(["group", "code"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(code_counts)


### 4.1 Unique Codes by Group


In [ ]:
codes_by_group = (
    df_analysis[["group", "code"]]
    .drop_duplicates()
    .sort_values(["group", "code"])
    .reset_index(drop=True)
)

display(codes_by_group)


### 4.2 Top Codes


In [ ]:
top_codes = code_counts.head(TOP_N_CODES).copy()

display(top_codes)


In [ ]:
plot_data = top_codes.sort_values("count", ascending=True)

fig, ax = plt.subplots(
    figsize=(7.0, 4.2),
    constrained_layout=True,
)

ax.barh(
    plot_data["code"],
    plot_data["count"],
)

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=8,
    )

plt.show()


### 4.3 Top Codes for Each Group


In [ ]:
top_codes_by_group = (
    code_counts
    .groupby("group", group_keys=False)
    .head(TOP_N_CODES_PER_GROUP)
    .copy()
)

display(top_codes_by_group)


In [ ]:
# Create a single horizontal chart while retaining the group-specific
# colors. The y-axis is built from all selected codes, so codes from
# different groups can be compared without overlapping bars.

plot_data = top_codes_by_group.sort_values(
    ["group", "count"],
    ascending=[True, True],
).copy()

fig_height = max(4.0, 0.38 * len(plot_data))

fig, ax = plt.subplots(
    figsize=(7.0, fig_height),
    constrained_layout=True,
)

y_positions = range(len(plot_data))

ax.barh(
    list(y_positions),
    plot_data["count"],
    color=[
        COLORS.get(group, "gray")
        for group in plot_data["group"]
    ],
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(plot_data["code"])

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for y, (_, row) in zip(y_positions, plot_data.iterrows()):
    ax.text(
        row["count"] + 0.15,
        y,
        str(row["count"]),
        va="center",
        fontsize=8,
    )

legend_handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS[group],
        label=label,
    )
    for group, label in GROUP_LABELS.items()
]

ax.legend(
    handles=legend_handles,
    title="Code Group",
    frameon=False,
    loc="lower right",
)

plt.show()


## 5. Themes


In [ ]:
# Theme frequencies are calculated from coded excerpts.
#
# A theme can contain multiple codes. Consequently, its frequency here
# represents the number of coded excerpts associated with that theme.

theme_counts = (
    df_analysis
    .groupby(["group", "theme"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(theme_counts)


### 5.1 Codes Associated with Each Theme


In [ ]:
theme_codes = (
    df_analysis
    .groupby(["group", "theme"], dropna=False)["code"]
    .agg(lambda values: sorted(set(values.dropna())))
    .reset_index()
)

theme_codes["code_count"] = theme_codes["code"].apply(len)

display(theme_codes)


### 5.2 Theme Ranking


In [ ]:
theme_ranking = theme_counts.sort_values(
    "count",
    ascending=False,
).copy()

display(theme_ranking)


In [ ]:
plot_data = theme_ranking.sort_values("count", ascending=True)

fig_height = max(4.0, 0.42 * len(plot_data))

fig, ax = plt.subplots(
    figsize=(7.0, fig_height),
    constrained_layout=True,
)

ax.barh(
    plot_data["theme"],
    plot_data["count"],
    color=[
        COLORS.get(group, "gray")
        for group in plot_data["group"]
    ],
)

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=8,
    )

plt.show()


### 5.3 Theme Ranking by Group


In [ ]:
# Prepare the three groups independently. This makes the plotting order
# deterministic and avoids the duplicated plotting logic in the original
# notebook.

group_data = {
    group: (
        theme_counts[theme_counts["group"] == group]
        .sort_values("count", ascending=True)
        .copy()
    )
    for group in GROUP_LABELS
}

for group, data in group_data.items():
    print(f"\n{GROUP_LABELS[group]}")
    display(data)


## 6. Main Thematic Analysis Figure


The following figure combines all themes and uses color to distinguish the
three thematic groups. The figure is saved as a PDF so it can be used as a
vector graphic in the paper.


In [ ]:
fig, ax = plt.subplots(
    figsize=(7.0, 4.8),
    constrained_layout=True,
)

plot_data = theme_counts.sort_values("count", ascending=True)

y_positions = range(len(plot_data))

ax.barh(
    list(y_positions),
    plot_data["count"],
    color=[
        COLORS.get(group, "gray")
        for group in plot_data["group"]
    ],
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(plot_data["theme"])

ax.set_xlabel("Frequency")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for y, (_, row) in zip(y_positions, plot_data.iterrows()):
    ax.text(
        row["count"] + 0.15,
        y,
        str(row["count"]),
        va="center",
        fontsize=8,
    )

legend_handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS[group],
        label=label,
    )
    for group, label in GROUP_LABELS.items()
]

ax.legend(
    handles=legend_handles,
    title="Theme Group",
    frameon=False,
    loc="lower right",
)

figure_path = OUTPUT_DIR / "thematic_analysis.pdf"

fig.savefig(
    figure_path,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {figure_path}")


## 7. Portuguese Version


This section creates a Portuguese version of the thematic-analysis figure
for presentation or documentation purposes.

The English analysis data are not modified; a separate copy is used for
translation.


In [ ]:
THEME_TRANSLATION = {
    "Growing acceptance of LLMs":
        "Crescente aceitação de LLMs",
    "Perceived Advantages of LLMs":
        "Vantagens percebidas dos LLMs",
    "Preference for Traditional Approaches":
        "Preferência por abordagens tradicionais",
    "Selective trust and human oversight":
        "Confiança seletiva e supervisão humana",
    "Learning and reflection":
        "Aprendizado e reflexão",
    "Maintenance-driven changes":
        "Mudanças motivadas por manutenção",
    "Quality and Technical Improvements":
        "Melhorias de qualidade e técnicas",
    "Automated transformations":
        "Transformações automatizadas",
    "Manual and incremental refactoring":
        "Refatoração manual e incremental",
    "Tool-assisted transformations":
        "Transformações assistidas por ferramentas",
}

theme_counts_pt = theme_counts.copy()

theme_counts_pt["theme"] = (
    theme_counts_pt["theme"]
    .replace(THEME_TRANSLATION)
)

display(theme_counts_pt)


In [ ]:
fig, ax = plt.subplots(
    figsize=(7.0, 4.8),
    constrained_layout=True,
)

plot_data = theme_counts_pt.sort_values("count", ascending=True)

y_positions = range(len(plot_data))

ax.barh(
    list(y_positions),
    plot_data["count"],
    color=[
        COLORS.get(group, "gray")
        for group in plot_data["group"]
    ],
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(plot_data["theme"])

ax.set_xlabel("Frequência")
ax.set_ylabel("")

ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for y, (_, row) in zip(y_positions, plot_data.iterrows()):
    ax.text(
        row["count"] + 0.15,
        y,
        str(row["count"]),
        va="center",
        fontsize=8,
    )

legend_handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS["motivation"],
        label="Motivação",
    ),
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS["tools"],
        label="Ferramentas e Automação",
    ),
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS["llm"],
        label="Uso de LLMs",
    ),
]

ax.legend(
    handles=legend_handles,
    title="Grupo de Temas",
    frameon=False,
    loc="lower right",
)

figure_path_pt = OUTPUT_DIR / "analise_tematica_pt.pdf"

fig.savefig(
    figure_path_pt,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {figure_path_pt}")


## 8. Export Aggregated Results


The following files contain only aggregated results derived from the
ATLAS.ti export. They are useful for checking the numerical results without
having to rerun the entire notebook.


In [ ]:
# Export code frequencies.
code_output = OUTPUT_DIR / "code_frequencies.csv"
code_counts.to_csv(code_output, index=False)

# Export theme frequencies.
theme_output = OUTPUT_DIR / "theme_frequencies.csv"
theme_counts.to_csv(theme_output, index=False)

# Export theme-to-code mapping.
theme_codes_output = OUTPUT_DIR / "theme_code_mapping.csv"
theme_codes.to_csv(theme_codes_output, index=False)

print("Generated files:")
for path in [
    code_output,
    theme_output,
    theme_codes_output,
    OUTPUT_DIR / "thematic_analysis.pdf",
    OUTPUT_DIR / "analise_tematica_pt.pdf",
]:
    print(f"  - {path}")


## 9. Summary


In [ ]:
print("Thematic analysis summary")
print("=" * 40)

print(f"Total coded excerpts: {len(df_analysis):,}")
print(f"Unique codes: {df_analysis['code'].nunique():,}")
print(f"Unique themes: {df_analysis['theme'].nunique():,}")
print(f"Theme groups: {df_analysis['group'].nunique():,}")

print("\nExcerpts by group:")
display(
    df_analysis["group"]
    .value_counts()
    .rename_axis("group")
    .to_frame("count")
)

print("\nThemes by frequency:")
display(
    theme_counts.sort_values(
        "count",
        ascending=False,
    )
)


## Reproducibility Notes

- The source qualitative coding was performed in **ATLAS.ti**.
- This notebook starts from the ATLAS.ti export and performs aggregation
  and visualization only.
- The input CSV should therefore be treated as the source artifact for
  this notebook.
- The generated tables and figures can be included in the replication
  package.
- If the dataset contains sensitive or identifiable developer information,
  the original ATLAS.ti export should not be published without appropriate
  anonymization and authorization.
